Essa primeira célula é a unica que precis ser sempre executada, as seguintes podem ser executadas de maneira unitária e na ordem desejada

In [1]:
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

import os
from sql_utils import drop_table, build_postgres_engine

engine = build_postgres_engine(
        "localhost",
        int(os.environ.get("POSTGRES_PORT", 5432)),
        os.environ["POSTGRES_DB"],
        os.environ["POSTGRES_USER"],
        os.environ["POSTGRES_PASSWORD"],
    )
    
download_path = project_root / "data" /"cache"
schema_name = "raw"
chunck_size = 100_000


# SENASP

In [ ]:
from dwl_utils.senasp import loop_download as dwl_senasp_loop

year = [2024,2023,2022,2021,2020,2019,2018,2017,2016,2015]
url = "https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/download/dnsp-base-de-dados/bancovde-{y}.xlsx/@@download/file"
table_name = "SENASP"

dwl_senasp_loop(url, year, download_path, schema_name, table_name, chunck_size)


é assim que printa uma tabela pra ver se baixou como deveria

In [ ]:
# import os

# table_name = "SENASP"

# print_table_head(engine, schema_name, table_name, rows=5)


# Disk 100 
nem colunas padronizadas tem

In [ ]:


# drop_table(schema_name, "disk100", engine)

In [ ]:
# from dwl_utils.disk100 import loop_download as dwl_disk100_loop

# periods = [
#            "segundo-semestre-de-2020", "primeiro-semestre-de-2020",
#            "segundo-semestre-de-2021", "primeiro-semestre-de-2021",
#            "segundo-semestre-de-2022", "primeiro-semestre-de-2022",
#            "segundo-semestre-de-2023", "primeiro-semestre-de-2023",
#            "segundo-semestre-de-2024", "primeiro-semestre-de-2024",
#            ]
# url = "https://www.gov.br/mdh/pt-br/acesso-a-informacao/dados-abertos/disque100/{y}"

# table_name = "disk100"

# dwl_disk100_loop(url, periods, download_path, schema_name, table_name, chunck_size)


In [ ]:
# import os

# from sql_utils import print_table_head, build_postgres_engine

# engine = build_postgres_engine(
#     "localhost",
#     int(os.environ.get("POSTGRES_PORT", 5432)),
#     os.environ["POSTGRES_DB"],
#     os.environ["POSTGRES_USER"],
#     os.environ["POSTGRES_PASSWORD"],
# )

# table_name = "disk100"

# print_table_head(engine, schema_name, table_name, rows=5)

# Agregado populacional por municipio

https://ftp.ibge.gov.br/Censos/leia_me.txt
https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Municipio_csv/
https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Municipio_csv/Agregados_por_municipios_demografia_BR.zip
https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/dicionario_de_dados_agregados_por_setores_censitarios_20250417.xlsx

Baixo os dados + legenda de códigos

In [ ]:
from dwl_utils import download_sheet_from_url
import pandas as pd

table_name = "APM_Demografico"

url_df = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Municipio_csv/Agregados_por_municipios_demografia_BR.zip"
df_path = download_sheet_from_url(url_df, "table", download_path, table_name, "zip")

url_leg = "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/dicionario_de_dados_agregados_por_setores_censitarios_20250417.xlsx"
leg_path = download_sheet_from_url(url_leg, "leg", download_path, table_name, "xlsx")

Leio os dados e ajusto no dataframe o nome das colunas com base na legenda

In [ ]:
from sql_utils import normalize_column_name

df = pd.read_csv(df_path, encoding="latin1", sep=";")

leg = pd.read_excel(leg_path, sheet_name="Dicionário não PCT")

leg=leg[leg["Tema"] == "Demografia"]
leg["Descrição"] = leg["Descrição"].apply(normalize_column_name)

mapping = dict(
    zip(leg["Variável"], leg["Descrição"])
)

df = df.rename(columns=mapping)
df = df.reset_index().rename(columns={"index": "id_tabela"})

Cria a tabela e insere no db

In [ ]:
from sql_utils import upload_dataframe_to_postgres, create_table_from_dataframe
from sqlalchemy import MetaData

create_table_from_dataframe(df, engine, MetaData(schema=schema_name), schema_name, table_name)
upload_dataframe_to_postgres(df, engine, schema_name, table_name, chunk_size=100_000)

# Dados municipais do PIB

https://ftp.ibge.gov.br/Pib_Municipios/2022_2023/base/base_de_dados_2010_2023_xlsx.zip

aqui também tem dados geograficos que podem ser úteis

In [ ]:
from dwl_utils import download_sheet_from_url
import pandas as pd

table_name = "Pib_municipal"

url_df = "https://ftp.ibge.gov.br/Pib_Municipios/2022_2023/base/base_de_dados_2010_2023_xlsx.zip"
df_path = download_sheet_from_url(url_df, "a", download_path, table_name, "zip")

In [ ]:
import zipfile

with zipfile.ZipFile(df_path) as z:
    print(z.namelist())
    
    with z.open((z.namelist())[0]) as f:
        df = pd.read_excel(f, engine="openpyxl")

limpando o dataframe do que a gente não precisa

In [ ]:
df = df[df["Ano"] >= 2015]
df = df.drop(
    columns=[
        "Amazônia Legal", 
        "Semiárido",
        "Valor adicionado bruto da Agropecuária, \na preços correntes\n(R$ 1.000)",
        "Valor adicionado bruto da Indústria,\na preços correntes\n(R$ 1.000)",
        "Valor adicionado bruto dos Serviços,\na preços correntes \n- exceto Administração, defesa, educação e saúde públicas e seguridade social\n(R$ 1.000)",
        "Valor adicionado bruto da Administração, defesa, educação e saúde públicas e seguridade social, \na preços correntes\n(R$ 1.000)",
        "Valor adicionado bruto total, \na preços correntes\n(R$ 1.000)",
        "Impostos, líquidos de subsídios, sobre produtos, \na preços correntes\n(R$ 1.000)",
    ])

df = df.reset_index().rename(columns={"index": "id_tabela"})

In [28]:
from sql_utils import upload_dataframe_to_postgres, create_table_from_dataframe
from sqlalchemy import MetaData

create_table_from_dataframe(df, engine, MetaData(schema=schema_name), schema_name, table_name)
upload_dataframe_to_postgres(df, engine, schema_name, table_name, chunk_size=100_000)

Inserindo raw.Pib_municipal: 100%|██████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.21s/chunk, li


50130

# População distribuição etária por municipio:

https://sidra.ibge.gov.br/tabela/9514#/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/6557,6653,49108,49109,60040,60041,93084,93085,93086,93087,93088,93089,93090,93091,93092,93093,93094,93095,93096,93097,93098,100362/c286/6556,113635/l/v,p+c2+c287,t+c286/cfg/cod,/resultado

In [ ]:
from dwl_utils.ibge_age_range import loop_dowload as dwl_age_range_loop

table_name = "age_range_ibge"

urls:dict = {
    "0_4": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93070/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "5_9" : "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93084/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "10_14": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93085/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "15_19": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93086/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "20_24": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93087/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "25_29": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93088/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "30_34": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93089/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "35_30": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93090/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "40_44": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93091/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "45_49": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93092/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "50_54": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93093/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "55_59": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93094/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "60_64": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93095/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "65_69": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93096/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "70_74": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93097/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "75_79": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/93098/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "80_84": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/49108/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "85_89": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/49109/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "90_94": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/60040/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "90_95": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/60041/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
    "100_mais": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela9514.xlsx&terr=NC&rank=-&query=t/9514/n1/all/n3/all/n6/all/v/allxp/p/all/c2/all/c287/6653/c286/6556,113635/l/v,p%2Bc2%2Bc287,t%2Bc286",
}

dwl_age_range_loop(urls, download_path, schema_name, table_name, engine, chunck_size=100_000)

Inserindo raw.age_range_ibge: 100%|██████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.14chunk/s, l


# Distribuição de renda por municipio
https://sidra.ibge.gov.br/Tabela/10292

In [52]:
from dwl_utils.ibge_income_dist import loop_dowload as dwl_income_dist

table_name = "dist_renda_ibge"

urls: dict = {
    "t_0_5": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela10292.xlsx&terr=N&rank=-&query=t/10292/n1/all/n3/all/n6/all/v/4090/p/all/c2/all/c11915/96177,96179,96180,96181,96185,99822,99823,99824/l/v,p%2Bc2,t%2Bc11915",
    "5_mais": "https://sidra.ibge.gov.br/geratabela?format=xlsx&name=tabela10292.xlsx&terr=N&rank=-&query=t/10292/n1/all/n3/all/n6/all/v/4090/p/all/c2/all/c11915/96182,96184,99825,99828/l/v,p%2Bc2,t%2Bc11915",
}

dwl_income_dist(urls, download_path, schema_name, table_name, engine, chunck_size=100_000)


Inserindo raw.dist_renda_ibge: 100%|██████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.07chunk/s, 
Inserindo raw.dist_renda_ibge: 100%|██████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.40chunk/s, 


===============================================

https://sidra.ibge.gov.br/pesquisa/pnadct/tabelas

Tabela Sidra n. 9517.
Tabela Sidra n. 10292

https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Mulheres/Tabela_4_Nivel_de_instrucao_e_existencia_de_deficiencia.xlsx
https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Mulheres/Tabela_10_Mulheres_com_filhos_nascidos_vivos.xlsx
https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Mulheres/Tabela_9_Tempo_de_deslocamento_para_o_trabalho.xlsx
https://ftp.ibge.gov.br/Indicadores_Sociais/Informacoes_Basicas_Municipais/tabela33.zip
https://ftp.ibge.gov.br/Indicadores_Sociais/Informacoes_Basicas_Municipais/tabela42a.zip
https://ftp.ibge.gov.br/Indicadores_Sociais/Informacoes_Basicas_Municipais/tabela42b.zip


Mt bom, mas dará um trabalho homerico pra tratar:
https://ftp.ibge.gov.br/Perfil_Municipios/2024/Tabelas_de_Resultados/

Dados da OCDE IMPORTANTISSIMOS:
https://data-explorer.oecd.org/?fs[0]=Topic%2C0|Education%20and%20skills%23EDU%23&pg=0&fc=Topic&bp=true&snb=113&isAvailabilityDisabled=false